In [ ]:
from pathlib import Path
import pickle

import pandas as pd
from tslearn.metrics import dtw
import numpy as np
from joblib import Parallel, delayed

In [ ]:
real = pd.read_csv(
    Path.home()
    / "Research"
    / "Virtual-ALS-Patients"
    / "data"
    / "no_labels_no_birth_year"
    / "train.csv"
)
real.head()

In [ ]:
# filter out patients with no temporal data (all unknowns)

dfs = []

for _, pdf in real.groupby(by="REF"):
    if (pdf[[f"P{j}" for j in range(1, 13)]] == -1.0).all().all():
        continue
    dfs.append(pdf)
    
real = pd.concat(dfs, ignore_index=True)

In [ ]:
SEED = 42

In [ ]:
version = "2025-03-28 15:06:33"
fname = f"synthetic_dfs_n_samples=100_patientflow_{version}_seed=42.pkl"

with open(fname, "rb") as f:
    gen_dfs, categorical_feats_info = pickle.load(f)

In [ ]:
for gen_df in gen_dfs:
    gen_df["rounded_Age_onset"] = gen_df["Age_onset"].astype(int).astype(float)

In [ ]:
# sample-to-population attack El Emam et al., 2020
# Age_onset Gender and Ethnicity are quasi-identifiers.

def sample_to_population(real_df: pd.DataFrame, gen_df: pd.DataFrame):
    acc = 0.0
    real_adj = pd.concat(list(map(lambda x: x[1][["Age_onset", "Gender", "Ethnicity"]].iloc[0].to_frame().T, real_df.groupby("REF"))), ignore_index=True)
    fake_adj = pd.concat(list(map(lambda x: x[1][["rounded_Age_onset", "Gender", "Ethnicity"]].iloc[0].to_frame().T, gen_df.groupby("REF"))), ignore_index=True)
    fake_adj = fake_adj.rename(columns={"rounded_Age_onset": "Age_onset"})

    for _, row in real_adj.iterrows():
        fs = len(
            real_adj[
                (real_adj["Age_onset"] == row["Age_onset"])
                & (real_adj["Gender"] == row["Gender"])
                & (real_adj["Ethnicity"] == row["Ethnicity"])
            ]
        )
        acc += (1 / fs) * bool(
            len(
                fake_adj[
                    (fake_adj["Age_onset"] == row["Age_onset"])
                    & (fake_adj["Gender"] == row["Gender"])
                    & (fake_adj["Ethnicity"] == row["Ethnicity"])
                ]
            )
        )

    return acc / len(real_adj)

parallel = Parallel(n_jobs=-1, verbose=10)
probs = np.array(parallel(delayed(sample_to_population)(real, gen_df) for gen_df in gen_dfs))
    
print(f"{probs.mean()} +/- {probs.std()}")

In [ ]:
probs.min(), probs.max()

In [ ]:
numerical_feats = ["Age_onset", "Height (m)", "Weight"]

categorical_feats = [
    "Gender",
    "Ethnicity",
    "NIV",
    "Tracheostomy",
    "PEG",
    "UMNvsLMN",
    "Onset",
    "Limb_O",
    "Limbs_Impairment",
    "Limbs_Side",
    "Weightloss_>10%",
    "ALS_familiar_history",
    "Ever_smoked",
    "Blood_hypertension",
    "Diabetes",
    "Dyslipidemia",
    "Thyroid",
    "Autoimmune",
    "SOD1 Mutation",
    "Stroke",
    "Cardiac_disease",
    "Primary_cancer",
    "C9orf72",
    "TARDBP mutation",
    "FUS mutation",
]

In [ ]:
# Pairwise distance between fake and real patients

def distance_metric(p1, p2, categorical_distance="binary"):
    """
    categorical_distance if binary:
        1 if p1 == p2 else 0
    categorical_distance if median:
        SMOTE-NC distance for categorical features
    """
    static_p1 = p1[numerical_feats + categorical_feats].iloc[0]
    temp_p1 = p1[[f"P{i}" for i in range(1, 13)]]
    static_p2 = p2[numerical_feats + categorical_feats].iloc[0]
    temp_p2 = p2[[f"P{i}" for i in range(1, 13)]]

    dist = np.sqrt(np.sum((static_p1[numerical_feats].to_numpy() - static_p2[numerical_feats].to_numpy()) ** 2))
    
    if categorical_distance == "binary":
        dist += np.sum(static_p1[categorical_feats].to_numpy() != static_p2[categorical_feats].to_numpy())
    
    dist += dtw(temp_p1.to_numpy(), temp_p2.to_numpy())
    
    return dist


In [ ]:
def patient_distance_matrix(real_df: pd.DataFrame, gen_df: pd.DataFrame):
    real_patients = [patient_df for _, patient_df in real_df.groupby(by="REF")]
    fake_patients = [patient_df for _, patient_df in gen_df.groupby(by="REF")]

    dists = np.zeros((len(real_patients), len(fake_patients)))

    indices = list(zip(*np.tril_indices(len(real_patients))))

    for i, j in indices:
        dists[i, j] = distance_metric(real_patients[i], fake_patients[j])
    
    return dists

In [ ]:
dists = list(parallel(delayed(patient_distance_matrix)(real, gen_df) for gen_df in gen_dfs))

In [ ]:
with open(f"privacy_dists_patientflow_{version}_seed={SEED}.pkl", "wb") as f:
    dists = pickle.dump(dists, f)

In [ ]:
with open(f"privacy_dists_patientflow_{version}_seed={SEED}.pkl", "rb") as f:
    dists = pickle.load(f)

In [ ]:
num_real_patients = len([patient_df for _, patient_df in real.groupby(by="REF")])

In [ ]:
means = np.array([_dists[np.tril_indices(num_real_patients)].mean() for _dists in dists])
means.mean()

In [ ]:
stds = np.array([_dists[np.tril_indices(num_real_patients)].std() for _dists in dists])
stds.mean()

In [ ]:
zero_distances = np.array([(_dists[np.tril_indices(num_real_patients)] == 0.0).sum() for _dists in dists])

In [ ]:
zero_distances